# Live technical demo — IS 455 final project

This notebook is the **in-class demo controller**. It runs the **same orchestrator** as the CLI (`python src/build.py`): `run_pipeline` from `src/build.py`, so stage order, logging, monitoring writes, and audit logic match the reproducible entrypoint **line-for-line**.

**After pipeline cells**, we inspect DuckDB evidence required by the FP contract: `runs`, `latency_logs`, `row_count_reconciliation`, `audit_results`, and a sample retrieval for the user-facing surface.


From the FP announcement:

- **Full rebuild** — one-command rebuild from raw; **idempotent** (same input → same output), verified with an **audit artifact**, not verbal assurance.
- **Augmented (incremental) build** — when new or corrected data arrives, re-process only what is needed; show **what was recomputed vs left alone** vs a fresh full rebuild.
- **Concrete update event** — run the augmented build **live** on a second batch (prepared file is fine): show delta.
- **Monitoring** — runs distinguishable by mode, scope, counts (e.g. `runs` table rows).
- Be ready to answer: grain (`one row = ?`), full vs augmented **this run**, recompute vs skip, **user-facing surface + refresh**, **failure modes** in raw and how they surface, **what fails at runtime** and how you detect it.


## Mirror of `run_pipeline` in `src/build.py`

| Order | Stage | `build.py` role |
|------:|--------|------------------|
| 1 | Setup | `duckdb.connect`, `monitoring.init_schema`, `monitoring.create_run` |
| 2 | INGEST | `ingest.ingest_raw_documents` + latency |
| 3 | CLEAN | `clean.validate_and_clean_raw` + latency |
| 4 | CURATE | `curate.curate_documents` + counts (new / updated / unchanged) |
| 5 | CHUNK | `chunk.chunk_new_documents` (delta docs only) |
| 6 | EMBED | `embed.embed_new_chunks` |
| 7 | INDEX | `index_ivf.build_index` |
| 8 | RECONCILIATION | `monitoring.write_reconciliation` |
| 9 | AUDIT | `audit.run_audit` — **only when `mode == "full"`** |
| 10 | Finish | `monitoring.finish_run` |

The cells below call **`run_pipeline(...)`** so stdout and behavior match the CLI.


In [15]:
import os
import sys
import subprocess
import importlib
from pathlib import Path


def ensure_module(module_name: str, pip_name: str | None = None):
    """Install missing modules on the fly (useful for fresh Colab runtimes)."""
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        target = pip_name or module_name
        print(f"[SETUP] Installing missing dependency: {target}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", target])


# Core deps required by this notebook
ensure_module("duckdb", "duckdb")
ensure_module("pandas", "pandas")
ensure_module("numpy", "numpy")
ensure_module("faiss", "faiss-cpu")
ensure_module("sentence_transformers", "sentence-transformers")
ensure_module("transformers", "transformers")

import duckdb
import pandas as pd
from IPython.display import display

# Robust project-root detection (works in local IDE and Colab)
def is_project_root(p: Path) -> bool:
    return (p / "src").is_dir() and (p / "data").is_dir()


def detect_project_root(start: Path) -> Path:
    current = start.resolve()

    # 1) Current directory + parents
    for candidate in [current, *current.parents]:
        if is_project_root(candidate):
            return candidate

    # 2) Common local + Colab locations
    home = Path.home()
    common_candidates = [
        home / "Desktop" / "455 Final",
        home / "455 Final",
        Path("/content") / "455 Final",
        Path("/content") / "drive" / "MyDrive" / "455 Final",
        current / "455 Final",
        current / "Desktop" / "455 Final",
    ]
    for candidate in common_candidates:
        if is_project_root(candidate):
            return candidate

    # 3) Shallow scans for a repo that contains this notebook
    scan_roots = [home / "Desktop", Path("/content"), Path("/content") / "drive" / "MyDrive"]
    for root in scan_roots:
        if not root.is_dir():
            continue
        for d in root.iterdir():
            if d.is_dir() and is_project_root(d) and (d / "notebooks" / "live_demo.ipynb").exists():
                return d

    raise RuntimeError(
        "Could not locate project root containing 'src/' and 'data/'. "
        "Set cwd to repository root (or /content/<repo>) and rerun this cell."
    )


PROJECT_ROOT = detect_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.build import run_pipeline
from src.retrieve import retrieve

# Relative paths only (FP requirement)
DATASET_NAME = "live_demo"
OUTPUT_DIR   = PROJECT_ROOT / "outputs" / "live_demo"
DB_PATH      = OUTPUT_DIR / "wiki_demo.duckdb"
INITIAL_JSONL = PROJECT_ROOT / "data" / "live_demo" / "initial_sample.jsonl"
UPDATE_JSONL  = PROJECT_ROOT / "data" / "live_demo" / "update_sample.jsonl"
INDEX_PATH    = OUTPUT_DIR / "faiss" / "index.faiss"

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)


def q(sql: str, db_path=None) -> pd.DataFrame:
    p = db_path or DB_PATH
    if not p.exists():
        print(f"Database not found: {p}")
        return pd.DataFrame()
    con = duckdb.connect(str(p), read_only=True)
    try:
        return con.execute(sql).df()
    finally:
        con.close()


print("PROJECT_ROOT :", PROJECT_ROOT)
print("DB_PATH      :", DB_PATH)
print("INITIAL_JSONL:", INITIAL_JSONL, "exists:", INITIAL_JSONL.exists())
print("UPDATE_JSONL :", UPDATE_JSONL, "exists:", UPDATE_JSONL.exists())

PROJECT_ROOT : C:\Users\Gawon Lim\Desktop\455 Final
DB_PATH      : C:\Users\Gawon Lim\Desktop\455 Final\outputs\live_demo\wiki_demo.duckdb
INITIAL_JSONL: C:\Users\Gawon Lim\Desktop\455 Final\data\live_demo\initial_sample.jsonl exists: True
UPDATE_JSONL : C:\Users\Gawon Lim\Desktop\455 Final\data\live_demo\update_sample.jsonl exists: True


In [16]:
if not INITIAL_JSONL.exists() or not UPDATE_JSONL.exists():
    print(
        "Prepare data/live_demo/initial_sample.jsonl and update_sample.jsonl before running the pipeline.\n"
        'See README "Live Demo Instructions" step 1 (make_live_demo_data).'
    )
else:
    print("Live demo JSONL files found.")

Live demo JSONL files found.


---
## A · Full rebuild (initial) — `mode="full"`

Equivalent CLI:

```bash
python src/build.py --mode full --dataset-name live_demo \
  --input data/live_demo/initial_sample.jsonl \
  --output outputs/live_demo --db outputs/live_demo/wiki_demo.duckdb --index-type ivf
```


In [17]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

run_id_a = run_pipeline(
    mode="full",
    dataset_name=DATASET_NAME,
    input_path=os.path.relpath(INITIAL_JSONL, PROJECT_ROOT),
    output_dir=os.path.relpath(OUTPUT_DIR, PROJECT_ROOT),
    db_path=os.path.relpath(DB_PATH, PROJECT_ROOT),
    index_type="ivf",
)
print("run_id (initial full):", run_id_a)


=== Run 1: FULL build [live_demo] ===
  Input:    data\live_demo\initial_sample.jsonl
  Output:   outputs\live_demo
  Database: outputs\live_demo\wiki_demo.duckdb

[SETUP] Loading tokenizer and model: intfloat/e5-base-v2
[SETUP] Embedding device: cuda


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6633.34it/s]


[SETUP] Model loaded.

[STAGE] INGEST
[INGEST] Reading from data\live_demo\initial_sample.jsonl


[INGEST] Loading documents: 100%|██████████| 40/40 [00:00<00:00, 20003.83it/s]

[INGEST] Ingested 40 documents

[STAGE] CLEAN
[CLEAN] Valid: 40, Rejected: 0

[STAGE] CURATE
[CURATE] New: 40, Updated: 0, Unchanged: 0
  New docs:       40
  Updated docs:   0
  Unchanged docs: 0

[STAGE] CHUNK



[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (4655 > 512). Running this sequence through the model will result in indexing errors


[CHUNK] Generated 613 chunks from 40 documents

[STAGE] EMBED
[EMBED] Embedding 613 chunks...


Batches: 100%|██████████| 20/20 [00:03<00:00,  5.35it/s]


[EMBED] Saved embeddings to outputs\live_demo\embeddings\run_1.npy
[EMBED] Inserted 613 embedding records

[STAGE] INDEX
[INDEX] Building IVF index with 613 vectors...
[INDEX] actual_nlist=61, num_vectors=613
[INDEX] Training IVF index...
[INDEX] Saved FAISS index to outputs\live_demo\faiss\index.faiss
[INDEX] Saved vector map to outputs\live_demo\faiss\vector_map.npy
[INDEX] Registered index 1908346e-d20b-4d98-b512-7117ff3622d3 in faiss_index_registry

[STAGE] RECONCILIATION

[STAGE] AUDIT
[AUDIT] No previous successful full run found for comparison

=== Run 1 Complete ===
  Status:           success
  Raw docs:         40
  Valid docs:       40
  Rejected:         0
  New docs:         40
  Updated docs:     0
  Unchanged docs:   0
  New chunks:       613
  Active chunks:    613
  New embeddings:   613
  Active embeds:    613
  Index vectors:    613
  Active docs:      40

run_id (initial full): 1


### Monitoring after run A — `runs`, `latency_logs`, `row_count_reconciliation`


In [18]:
display(q("""
SELECT run_id, mode, dataset_name, input_path, status,
       raw_doc_count, stg_valid_count, rejected_doc_count,
       new_doc_count, updated_doc_count, unchanged_doc_count,
       active_doc_count, active_chunk_count, active_embedding_count, index_vector_count,
       start_time, end_time, notes
FROM runs ORDER BY run_id DESC LIMIT 3
"""))

display(q("""
SELECT run_id, stage_name, duration_seconds, row_count, start_time, end_time
FROM latency_logs WHERE run_id = (SELECT MAX(run_id) FROM runs)
ORDER BY start_time
"""))

display(q("""
SELECT * FROM row_count_reconciliation
WHERE run_id = (SELECT MAX(run_id) FROM runs)
"""))

,run_id,mode,dataset_name,input_path,status,raw_doc_count,stg_valid_count,rejected_doc_count,new_doc_count,updated_doc_count,unchanged_doc_count,active_doc_count,active_chunk_count,active_embedding_count,index_vector_count,start_time,end_time,notes
0,1,full,live_demo,data\live_demo\initial_sample.jsonl,success,40,40,0,40,0,0,40,613,613,613,2026-05-03 23:19:53.895925,2026-05-03 23:20:02.096234,None


,run_id,stage_name,duration_seconds,row_count,start_time,end_time
0,1,ingest,0.067202,40,2026-05-03 23:19:55.672485,2026-05-03 23:19:55.739687
1,1,clean,0.048000,40,2026-05-03 23:19:55.742687,2026-05-03 23:19:55.790687
2,1,curate,0.076010,40,2026-05-03 23:19:55.793688,2026-05-03 23:19:55.869698
3,1,chunk,1.780495,613,2026-05-03 23:19:55.872697,2026-05-03 23:19:57.653192
4,1,embed,4.386040,613,2026-05-03 23:19:57.657193,2026-05-03 23:20:02.043233
5,1,index,0.038002,613,2026-05-03 23:20:02.046232,2026-05-03 23:20:02.084234


,run_id,raw_documents,clean_documents_active,rejected_documents,active_chunks,active_embeddings,index_vectors,reconciliation_status,details
0,1,40,40,0,613,613,613,PASS,"{""raw"": 40, ""clean_active"": 40, ""rejected"": 0, ""chunks"": 613, ""embeddings"": 613, ""index_vectors""..."


---
## B · Full rebuild again (same input) — idempotency + audit

Same command as A. Expect **`unchanged_doc_count`** aligned with all curated docs and **no duplicate work** in chunk/embed for unchanged docs. On the **second** successful full run for the same `(dataset_name, input_path)`, `build.py` runs **`audit.run_audit`** vs the previous full run.

Equivalent CLI: repeat Step 2 from README Live Demo.


In [19]:
run_id_b = run_pipeline(
    mode="full",
    dataset_name=DATASET_NAME,
    input_path=os.path.relpath(INITIAL_JSONL, PROJECT_ROOT),
    output_dir=os.path.relpath(OUTPUT_DIR, PROJECT_ROOT),
    db_path=os.path.relpath(DB_PATH, PROJECT_ROOT),
    index_type="ivf",
)
print("run_id (repeat full):", run_id_b)


=== Run 2: FULL build [live_demo] ===
  Input:    data\live_demo\initial_sample.jsonl
  Output:   outputs\live_demo
  Database: outputs\live_demo\wiki_demo.duckdb

[SETUP] Loading tokenizer and model: intfloat/e5-base-v2
[SETUP] Embedding device: cuda


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7106.93it/s]


[SETUP] Model loaded.

[STAGE] INGEST
[INGEST] Reading from data\live_demo\initial_sample.jsonl


[INGEST] Loading documents: 100%|██████████| 40/40 [00:00<00:00, 13348.09it/s]

[INGEST] Ingested 40 documents

[STAGE] CLEAN
[CLEAN] Valid: 40, Rejected: 0

[STAGE] CURATE
[CURATE] New: 0, Updated: 0, Unchanged: 40
  New docs:       0
  Updated docs:   0
  Unchanged docs: 40

[STAGE] CHUNK

[STAGE] EMBED
[EMBED] No missing active-chunk embeddings for run 2

[STAGE] INDEX
[INDEX] Building IVF index with 613 vectors...
[INDEX] actual_nlist=61, num_vectors=613
[INDEX] Training IVF index...
[INDEX] Saved FAISS index to outputs\live_demo\faiss\index.faiss
[INDEX] Saved vector map to outputs\live_demo\faiss\vector_map.npy
[INDEX] Registered index 063a09b7-b76e-4380-b547-fcd625f2db53 in faiss_index_registry

[STAGE] RECONCILIATION

[STAGE] AUDIT
[AUDIT] Comparing run 1 vs run 2
[AUDIT] active_doc_count: PASS | {'run_a': 40, 'run_b': 40}
[AUDIT] active_chunk_count: PASS | {'run_a': 613, 'run_b': 613}
[AUDIT] active_embedding_count: PASS | {'run_a': 613, 'run_b': 613}
[AUDIT] index_vector_count: PASS | {'run_a': 613, 'run_b': 613}


[AUDIT] chunk_hash_checksum: PASS | {'checksum_a': '89edb20396e08e49...', 'checksum_b': '89edb20396e08e49...'}
[AUDIT] Completed audit between run 1 and run 2

=== Run 2 Complete ===
  Status:           success
  Raw docs:         40
  Valid docs:       40
  Rejected:         0
  New docs:         0
  Updated docs:     0
  Unchanged docs:   40
  New chunks:       0
  Active chunks:    613
  New embeddings:   0
  Active embeds:    613
  Index vectors:    613
  Active docs:      40

run_id (repeat full): 2


### `audit_results` — idempotency evidence (full vs full)


In [20]:
display(q("SELECT * FROM audit_results ORDER BY audit_id DESC LIMIT 10"))
display(q("""
SELECT run_id, mode, new_doc_count, updated_doc_count, unchanged_doc_count, notes
FROM runs ORDER BY run_id DESC LIMIT 5
"""))

,audit_id,run_id_a,run_id_b,audit_type,result,details,created_at
0,cdae7ac0-892a-4739-9a40-e2ac287c7999,1,2,active_doc_count,PASS,"{""run_a"": 40, ""run_b"": 40}",2026-05-03 23:20:12.780302
1,6dbe14fb-1c47-4e18-be76-dcd78f60120c,1,2,chunk_hash_checksum,PASS,"{""checksum_a"": ""89edb20396e08e49..."", ""checksum_b"": ""89edb20396e08e49...""}",2026-05-03 23:20:12.780302
2,22c1976b-e9b7-4032-be92-3807c9b93413,1,2,active_embedding_count,PASS,"{""run_a"": 613, ""run_b"": 613}",2026-05-03 23:20:12.780302
3,1a7a2503-561a-4589-8083-520aad43e818,1,2,active_chunk_count,PASS,"{""run_a"": 613, ""run_b"": 613}",2026-05-03 23:20:12.780302
4,0f79d488-1bf8-4ee4-9ec5-bd5b36b6172b,1,2,index_vector_count,PASS,"{""run_a"": 613, ""run_b"": 613}",2026-05-03 23:20:12.780302


,run_id,mode,new_doc_count,updated_doc_count,unchanged_doc_count,notes
0,2,full,0,0,40,None
1,1,full,40,0,0,None


---
## C · Augmented build — `mode="augmented"`, update batch

Demonstrates the **incremental contract**: only new/changed documents are re-chunked and re-embedded; FAISS is rebuilt from **all** active embeddings (see README — IVF training rationale). Evidence: `new_doc_count`, `updated_doc_count`, `unchanged_doc_count` on this run and `latency_logs` row counts.

Equivalent CLI:

```bash
python src/build.py --mode augmented --dataset-name live_demo \
  --input data/live_demo/update_sample.jsonl \
  --output outputs/live_demo --db outputs/live_demo/wiki_demo.duckdb --index-type ivf
```


In [7]:
run_id_c = run_pipeline(
    mode="augmented",
    dataset_name=DATASET_NAME,
    input_path=os.path.relpath(UPDATE_JSONL, PROJECT_ROOT),
    output_dir=os.path.relpath(OUTPUT_DIR, PROJECT_ROOT),
    db_path=os.path.relpath(DB_PATH, PROJECT_ROOT),
    index_type="ivf",
)
print("run_id (augmented):", run_id_c)


=== Run 4: AUGMENTED build [live_demo] ===
  Input:    data\live_demo\update_sample.jsonl
  Output:   outputs\live_demo
  Database: outputs\live_demo\wiki_demo.duckdb

[SETUP] Loading tokenizer and model: intfloat/e5-base-v2
[SETUP] Embedding device: cuda


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6862.23it/s]


[SETUP] Model loaded.

[STAGE] INGEST
[INGEST] Reading from data\live_demo\update_sample.jsonl


[INGEST] Loading documents: 100%|██████████| 4/4 [00:00<00:00, 4113.07it/s]
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (4668 > 512). Running this sequence through the model will result in indexing errors


[INGEST] Ingested 4 documents

[STAGE] CLEAN
[CLEAN] Valid: 3, Rejected: 1

[STAGE] CURATE
[CURATE] New: 0, Updated: 1, Unchanged: 2
  New docs:       0
  Updated docs:   1
  Unchanged docs: 2

[STAGE] CHUNK
[CHUNK] Generated 15 chunks from 1 documents

[STAGE] EMBED
[EMBED] Embedding 15 chunks...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.67it/s]

[EMBED] Saved embeddings to outputs\live_demo\embeddings\run_4.npy
[EMBED] Inserted 15 embedding records

[STAGE] INDEX
[INDEX] Building IVF index with 641 vectors...
[INDEX] actual_nlist=64, num_vectors=641
[INDEX] Training IVF index...
[INDEX] Saved FAISS index to outputs\live_demo\faiss\index.faiss
[INDEX] Saved vector map to outputs\live_demo\faiss\vector_map.npy
[INDEX] Registered index cb3016fd-8975-4a56-9c91-dd057810a59e in faiss_index_registry

[STAGE] RECONCILIATION

=== Run 4 Complete ===
  Status:           success
  Raw docs:         4
  Valid docs:       3
  Rejected:         1
  New docs:         0
  Updated docs:     1
  Unchanged docs:   2
  New chunks:       15
  Active chunks:    641
  New embeddings:   15
  Active embeds:    641
  Index vectors:    641
  Active docs:      41

run_id (augmented): 4


### Compare runs — full vs augmented on this database


In [8]:
display(q("""
SELECT run_id, mode, input_path,
       new_doc_count, updated_doc_count, unchanged_doc_count,
       new_chunk_count, new_embedding_count, index_vector_count, notes
FROM runs ORDER BY run_id DESC LIMIT 6
"""))

display(q("SELECT * FROM faiss_index_registry ORDER BY created_at DESC LIMIT 5"))

,run_id,mode,input_path,new_doc_count,updated_doc_count,unchanged_doc_count,new_chunk_count,new_embedding_count,index_vector_count,notes
0,4,augmented,data\live_demo\update_sample.jsonl,0,1,2,15,15,641,FAISS rebuilt from all active embeddings; ETL-level delta processing confirmed by new/updated co...
1,3,full,data\live_demo\initial_sample.jsonl,0,0,40,0,0,641,NaN
2,2,full,data\live_demo\initial_sample.jsonl,38,1,1,594,594,641,NaN
3,1,augmented,data/live_demo/update_sample.jsonl,3,0,0,62,62,62,FAISS rebuilt from all active embeddings; ETL-level delta processing confirmed by new/updated co...


,index_id,run_id,dataset_name,index_type,embedding_model,metric,nlist,nprobe,num_vectors,index_path,vector_map_path,index_checksum,build_mode,is_active,created_at
0,cb3016fd-8975-4a56-9c91-dd057810a59e,4,live_demo,IVFFlat,intfloat/e5-base-v2,INNER_PRODUCT,64,10,641,outputs\live_demo\faiss\index.faiss,outputs\live_demo\faiss\vector_map.npy,fa71061263d561853716a7045b790d7635c314b172d147af46dcdbabd8bb9f41,augmented,True,2026-05-03 23:13:09.882210
1,661f363b-dfc9-4374-93bb-fb6f25a97311,3,live_demo,IVFFlat,intfloat/e5-base-v2,INNER_PRODUCT,64,10,641,outputs\live_demo\faiss\index.faiss,outputs\live_demo\faiss\vector_map.npy,7b5cf520f17e4320b194950239039a68c85c67046b75daa943194842db08cf20,full,False,2026-05-03 23:11:46.601311
2,c75512f3-8ad0-47b8-b3c7-5011b1b8a785,2,live_demo,IVFFlat,intfloat/e5-base-v2,INNER_PRODUCT,64,10,641,outputs\live_demo\faiss\index.faiss,outputs\live_demo\faiss\vector_map.npy,7b5cf520f17e4320b194950239039a68c85c67046b75daa943194842db08cf20,full,False,2026-05-03 23:10:43.167636
3,322d1bfb-b419-4fab-a0c6-9edb27d035e6,1,live_demo,IVFFlat,intfloat/e5-base-v2,INNER_PRODUCT,6,10,62,outputs/live_demo\faiss\index.faiss,outputs/live_demo\faiss\vector_map.npy,a7d39510814ed4c5fbc60a3cbc8ef412ab32a20cc5e459455f546998be75c6cd,augmented,False,2026-05-03 22:49:01.223547


## Six syllabus questions 

1. **One row = ?** — Curated analytical grain: **`fact_chunks`**: one row per active chunk (sliding window over cleaned article text); embeddings are one row per chunk in `fact_embeddings`.
2. **Full vs augmented on this run** — Full reloads from the given JSONL into raw and walks the same stages; augmented uses the **update file** and ETL marks **new / updated / unchanged** in curation.
3. **What is recomputed vs skipped** — Unchanged docs: **skip** chunk/embed; new/updated: **new** chunks and embeddings; index step rebuilds FAISS from all active vectors.
4. **Who sees what** — Downstream consumer: RAG or analyst using retrieval; **surface**: ranked chunk table from `retrieve.py` / embedding+FAISS; **refresh**: rebuild augmented when `update_sample.jsonl` is processed; staleness: **latest successful run** in `runs` and `faiss_index_registry`.
5. **Failure modes in raw** — e.g. empty text → `rejected_documents` with `reason_code`; duplicates → idempotent **unchanged**; trace in monitoring tables.
6. **Runtime failures** — Pipeline exceptions → `runs.status='failed'`; reconciliation failures visible in `row_count_reconciliation`; audit checks in `audit_results`.

Run the next cell to print **counts from the latest run** for quick Q&A.


In [9]:
display(q("""
SELECT reason_code, COUNT(*) AS n
FROM rejected_documents
WHERE run_id = (SELECT MAX(run_id) FROM runs)
GROUP BY reason_code
"""))
display(q("SELECT * FROM rejected_documents ORDER BY run_id DESC LIMIT 10"))

,reason_code,n
0,EMPTY_TEXT,1


,run_id,doc_id,title,reason_code,reason_detail,raw_snapshot
0,4,corrupted_doc_001,Corrupted Test Document,EMPTY_TEXT,text is empty or None after strip,"{""id"": ""corrupted_doc_001"", ""url"": """", ""title"": ""Corrupted Test Document"", ""text"": null}"
1,1,corrupted_doc_001,Corrupted Test Document,EMPTY_TEXT,text is empty or None after strip,"{""id"": ""corrupted_doc_001"", ""url"": """", ""title"": ""Corrupted Test Document"", ""text"": null}"


---
## D · User-facing surface — dense retrieval (live)

Same behavior as:

`python src/retrieve.py --db outputs/live_demo/wiki_demo.duckdb --index outputs/live_demo/faiss/index.faiss --query "..." --top-k 5`


In [10]:
if INDEX_PATH.exists() and DB_PATH.exists():
    demo_query = "What document discusses vector databases and retrieval systems?"
    df_hit = retrieve(
        query=demo_query,
        db_path=os.path.relpath(DB_PATH, PROJECT_ROOT),
        index_path=os.path.relpath(INDEX_PATH, PROJECT_ROOT),
        top_k=5,
    )
    display(df_hit)
else:
    print("Run pipeline cells above first (index and DB required).")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7654.19it/s]


,rank,score,chunk_id,doc_id,chunk_text,chunk_index,start_tok,end_tok,title,url
0,1,0.771135,0c4ddafa61c3f612d9ca1d010311b831dc142aa1de7fcec1195f2387266e7c92,1260,"for instance, by the paper on chosen - key - relations - in - the - middle attacks on aes - 128 ...",8,2560,2944,Advanced Encryption Standard,https://en.wikipedia.org/wiki/Advanced%20Encryption%20Standard
1,2,0.768275,7637e9542c94f356653ccf2edb800276a32ca22171c13aaceb502449c6ca66cd,1260,specific approval by the nsa is not deemed secure by the us government and cannot be used to pro...,13,4160,4544,Advanced Encryption Standard,https://en.wikipedia.org/wiki/Advanced%20Encryption%20Standard
2,3,0.766665,9dadabe01248538c4e3952a3045a7ed9727d4dc6d3f787f9fea16d9cd896e4a4,1201,"in 1974 by valerie sutton, is the first writing system to gain use among the public and the firs...",14,4480,4864,American Sign Language,https://en.wikipedia.org/wiki/American%20Sign%20Language
3,4,0.766418,5bc94ac0842668efefa174d7719fad6ed0a356e5dc2f56639744bed6b4cdb485,1260,15 gib / s on an i7 - 12700k ). implementations see also aes modes of operation disk encryption ...,14,4480,4668,Advanced Encryption Standard,https://en.wikipedia.org/wiki/Advanced%20Encryption%20Standard
4,5,0.757166,05c9572949c67eb1ea7e14f81eb4a7cd77fcb4a94dae7573ed4c428d8ca1f911,1271,"by "" public subscription "" to enable serious historical and academic study of babbage's plans, w...",4,1280,1664,Analytical engine,https://en.wikipedia.org/wiki/Analytical%20engine
